# Stage 2 - metamorphic expansion

Eighteen follow-up cases derived from eight of the eleven seeds - two each,
and three from `scheduling_move_tram` and `venue_lookup_seafood_lisbon` - with
one relation each from the catalogue its function permits (D22). The budget is
D21's, extended by D50, D51 and D61 wherever a case was the only way to reach a
tool or a relation.

Three seeds take no follow-up at all. The two weather-forecast seeds and the
surf-conditions seed exist to reach tools nothing else could (D51, D54), and a
perturbation of them would buy no coverage the suite does not already have.

A follow-up is a seed with its request **perturbed** and an **expected delta**
stated in the same constraint vocabulary the seed uses. The perturbation and
the delta are chosen by hand and live in `cases/derivations.py`; the shape of
the derived case - what it inherits, which predicates the relation implies,
that it runs at n = 4 where a seed runs at n = 8, and that it names its base -
is `src/metamorphic.py`.

This notebook applies them, shows each follow-up beside its seed, and reports
what a run of all twenty-nine produced: 29 cases, 160 executions at full n.

## What the relations can and cannot claim

Two things were settled before any of these were written, and both change what
is worth asserting.

**`search_places` is capped at five results per call.** So "ask for something
more specific and expect fewer results" - the obvious directional relation for
venue lookup - is not a relation this system has. A tighter constraint makes
the agent search *again*, and `places_surfaced` goes up. What narrowing
actually changes is *which* venues come back, and asking for a second kind of
place changes the shape of the answer outright. The suite asserts those two
and not the count.

**Half the suite has no structural delta, by construction.** D22 gives factual
QA only DIR-1 besides invariance, and advisory invariance alone. That is seven
of the eleven seeds and eight of the eighteen follow-ups: the same tools run,
the same nothing is written, and what changed is the *content* of the answer,
which only the judge sees. Their structural verdict is inherited from the seed
and will read `none` whether or not the agent answered the new question. That
is not a defect in the cases; it is where structural verification stops and
scoring starts.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parents[1]))  # the subtree, so `src` imports

import json
from collections import Counter

import yaml

from src.cases import CASES_DIR, load_cases, suite_version
from src.metamorphic import ELIGIBLE, RELATION_NAMES, derive
from cases.derivations import DERIVATIONS

## Deriving the eighteen

Regenerated from the seeds every time this runs, so a seed edit cannot leave a stale follow-up behind.

In [2]:
seeds = {c.case_id: c for c in load_cases() if c.provenance == "seeded"}
# Two follow-ups per seed (D21), with two exceptions the suite records
# (D50, D51): `scheduling_move_tram` takes three, because `schedule_block`
# and `unschedule_block` are reachable only under /reschedule and so both
# cases that reach them must derive from that seed; and
# `factual_qa_lisbon_surf_conditions` takes none, because it exists to
# reach `get_climate_normals` and a perturbation of it would buy no tool
# the suite does not already have.
assert set(DERIVATIONS) <= set(seeds)

written = []
for seed_id, pair in DERIVATIONS.items():
    for d in pair:
        data = derive(seeds[seed_id], d)
        path = CASES_DIR / f"{data['case_id']}.yaml"
        path.write_text(
            yaml.dump(data, sort_keys=False, allow_unicode=True,
                      default_flow_style=False, width=100),
            encoding="utf-8",
        )
        written.append(data["case_id"])

cases = load_cases()
expanded = [c for c in cases if c.provenance == "metamorphic"]
print(f"{len(written)} follow-ups derived; {len(cases)} cases in the suite")
print(f"executions per run: {sum(c.repetitions_n for c in cases)}  (D21 budget: 128)")
print(f"relations: {dict(sorted(Counter(c.metamorphic_relation for c in expanded).items()))}")
print(f"suite_version: {suite_version()[:16]}…")

18 follow-ups derived; 29 cases in the suite
executions per run: 160  (D21 budget: 128)
relations: {'DIR-1': 5, 'DIR-2': 3, 'DIR-3': 1, 'DIR-4': 2, 'INV-1': 4, 'INV-2': 1, 'INV-3': 1, 'INV-4': 1}
suite_version: 0c6068ee93fb1c08…


## Eligibility - why the relations are applied unevenly

The raw counts look arbitrary (five DIR-1, one DIR-3, one INV-2), so the first
thing to say is what the denominator is. A relation is applied only where it
*means* something, and D22 fixes that per function. Two examples make the rule
concrete.

**Eligible - DIR-2 (day changed) on `scheduling_move_tram`.** The seed asks to
move the tram ride to Saturday afternoon; the follow-up asks for Sunday, and
the case's `slot_scheduled` check moves with it. The perturbation names a
different day and the expected outcome *is* a different day, so the follow-up
makes a claim that can fail in one specific, checkable way.

**Not eligible - DIR-2 on `factual_qa_alfama_history`.** The seed asks what
happened in Alfama in the 1755 earthquake. There is no day in the request to
change, and no answer that a changed day would make different. The follow-up's
expected delta would be undefined, and a case that cannot state what ought to
differ is not a test - it is a second copy of the seed wearing a relation's
name.

So the opportunities are uneven before any choice is made: DIR-2, DIR-3 and
DIR-4 are eligible on 4 of the 11 seeds, while INV-1 to INV-4 are eligible on
all 11.

**Within eligibility there is a preference, and it is deliberate.** Directional
relations are used at 25-75% of their opportunity, invariance at 9-36%. A
directional relation asserts the answer *changes*, which fails specifically; an
invariance relation asserts sameness, which is the thing a language model is
most reliably good at. A suite weighted toward invariance would spend its
budget confirming robustness it already has, and report that as coverage.

**The residual limitation, stated rather than dressed up.** INV-2, INV-3 and
INV-4 carry one case each. That is a floor chosen so the catalogue is
instantiated rather than claimed - nothing statistical may be read off a single
case, and the table below says so itself.

In [3]:
from src import tables

# The grid, not a list per function: the question a reader has is "why does
# THIS relation have so few cases", and that is a column.
tables.eligibility()

,perturbation,advisory,factual_qa,scheduling,venue_lookup,eligible seeds
relation,,,,,,
DIR-1,constraint added or changed,-,yes,yes,yes,9
DIR-2,day changed,-,-,yes,yes,4
DIR-3,count changed,-,-,yes,yes,4
DIR-4,negation,-,-,yes,yes,4
INV-1,paraphrase,yes,yes,yes,yes,11
INV-2,typos and casing,yes,yes,yes,yes,11
INV-3,irrelevant detail added,yes,yes,yes,yes,11
INV-4,language switch (German),yes,yes,yes,yes,11


In [4]:
# Use against opportunity, not against the other relations (D61).
# DIR-2/3/4 look starved only until you read their denominator.
tables.relations()

,perturbation,kind,eligible seeds,follow-ups,share of opportunity
relation,,,,,
DIR-1,constraint added or changed,directional,9,5,0.556
DIR-2,day changed,directional,4,3,0.750
DIR-3,count changed,directional,4,1,0.250
DIR-4,negation,directional,4,2,0.500
INV-1,paraphrase,invariance,11,4,0.364
INV-2,typos and casing,invariance,11,1,0.091
INV-3,irrelevant detail added,invariance,11,1,0.091
INV-4,language switch (German),invariance,11,1,0.091


In [5]:
# The limitation travels with the numbers rather than living only in prose.
from src.metamorphic import relation_report

print(relation_report().split("\n\n", 1)[-1])

single instance: INV-2, INV-3, INV-4 — instantiated and exercised, but one case is not a sample. Nothing statistical may be claimed per-relation for these (D61).


## The tool-coverage metric, and why the suite grew past its budget

D21 fixed the budget at eight seeds and two follow-ups each. The suite is
eleven and eighteen. Every case beyond the budget was admitted by one metric,
and it is worth stating precisely, because "we added more tests" is not an
argument.

**The metric.** For each tool the agent has, count three different things:

| column | question it answers |
|---|---|
| `cases requiring` | how many cases assert the tool actually **ran** |
| `cases permitting` | how many merely allow it (`min: 0`) - a probe |
| `calls` / `errors` | what a real run did with it |

The gap between the first two columns is the whole point. A tool permitted by
eleven cases and required by none looks thoroughly covered and is not covered
at all: no case fails if it disappears. `tools_any_of` is read as the
disjunction it is - a two-element `any_of` requires *neither* tool
individually, because either one satisfies the case (D66).

**The admission rule.** A case beyond D21's budget earns its place only by
being the **only route** to something: a tool no other case requires, or a
relation no other case instantiates. A case that reaches only what the suite
already reaches is a second sample of a measurement it already has, and
raising `n` is the cheaper way to buy that. The table below is that rule made
checkable - for every extra case, what would be lost if it were deleted.

In [6]:
# The admission rule, made checkable: what would be lost if each case went.
# 'jointly with N' means N other cases also require it — delete them all and
# the tool is uncovered, which is still the only route to it (D69).
u = tables.unique_contribution()
u[u['carries coverage']][['provenance', 'route to', 'sole coverage']]

,provenance,route to,sole coverage
case,,,
advisory_packing_list__inv3_irrelevant_detail,metamorphic,relation INV-3,True
advisory_sintra_full_or_half_day__inv4_german,metamorphic,relation INV-4,True
factual_qa_alfama_history,seeded,read_wikipedia (jointly with 2),False
factual_qa_alfama_history__dir1_the_earthquake,metamorphic,read_wikipedia (jointly with 2),False
factual_qa_alfama_history__inv1_paraphrase,metamorphic,read_wikipedia (jointly with 2),False
factual_qa_lisbon_forecast_next_days,seeded,get_weather (jointly with 1),False
factual_qa_lisbon_surf_conditions,seeded,get_climate_normals,True
factual_qa_lisbon_trip_forecast,seeded,get_weather (jointly with 1),False
scheduling_belem_morning,seeded,propose_plan (jointly with 2),False


## Each follow-up beside its seed

The request, what changed, and the delta the case claims.

In [7]:
def show(seed, follow):
    checks_seed = seed.declared_constraints.checks.model_dump(exclude_none=True)
    checks_follow = follow.declared_constraints.checks.model_dump(exclude_none=True)
    changed = {k: v for k, v in checks_follow.items() if checks_seed.get(k) != v}
    dropped = [k for k in checks_seed if k not in checks_follow]

    print("=" * 100)
    print(f"{follow.case_id}")
    print(f"  {follow.metamorphic_relation} — {RELATION_NAMES[follow.metamorphic_relation]}"
          f"   (base: {seed.case_id}, n={follow.repetitions_n})")
    print(f"  seed asked   : {seed.opening_turn}")
    print(f"  now asks     : {follow.opening_turn}")
    if changed:
        print("  delta claimed:")
        for k, v in changed.items():
            print(f"      {k} = {json.dumps(v, ensure_ascii=False)}")
    else:
        print("  delta claimed: none structurally — the change is in the answer, and the judge scores it")
    if dropped:
        print(f"  dropped      : {dropped}  (the perturbation makes these untrue)")
    if follow.declared_constraints.predicates:
        print(f"  predicates   : {follow.declared_constraints.predicates}")
    print(f"  why          : {follow.notes}")


by_id = {c.case_id: c for c in cases}
for seed_id, pair in DERIVATIONS.items():
    for d in pair:
        show(seeds[seed_id], by_id[f"{seed_id}__{d.suffix}"])

venue_lookup_seafood_lisbon__dir1_and_a_hotel
  DIR-1 — constraint added or changed   (base: venue_lookup_seafood_lisbon, n=4)
  seed asked   : /find good seafood places near Cais do Sodré, nothing too pricey
  now asks     : /find good seafood places near Cais do Sodré, nothing too pricey — and somewhere to stay nearby too
  delta claimed:
      places_surfaced = {"min": 2}
      places_named = {"min": 3}
      places_named_types = ["restaurant", "lodging"]
  why          : The strongest perturbation available to a venue-lookup case: asking for a second kind of place changes the shape of the answer rather than its wording. Chosen over the obvious 'ask for something narrower and expect fewer results', which is not a relation this system has — search_places is capped at five per call, so narrowing pushes the agent to search again and the count rises.
venue_lookup_seafood_lisbon__dir4_not_a_meal
  DIR-4 — negation   (base: venue_lookup_seafood_lisbon, n=4)
  seed asked   : /find good sea

## What a run of all twenty-nine produced

One repetition each, so every follow-up has its base execution at the same
repetition to compare against (which is what the predicates need).

Not a baseline: `n = 1`, no judge, and the run row says so.

In [8]:
from src.checker import check_run
from src import execution as exmod
from src.db import admin

# Pinned, not "the newest run". The outputs stored in this notebook describe
# ONE run, and a selector that drifts would leave the prose describing numbers
# that are no longer below it. Re-point it deliberately when re-running.
RUN_ID = "09f288e9-b39b-4308-a66f-d769b2c05d56"

run_id = RUN_ID
meta = admin().table("study_runs").select(
    "notes, started_at, suite_version, api_version, api_commit, git_sha"
).eq("id", run_id).execute().data[0]
print(f"run {run_id[:8]}  {meta['started_at'][:19]}")
print(f"  {meta['notes']}")
print(f"  suite {meta['suite_version'][:12]}  api {meta['api_version']} @ {meta['api_commit']}")

run 09f288e9  2026-09-22T19:26:23
  FULL SUITE n=1 post-D59..D65 - retry policy, INV-2, versions, tables
  suite 1419b4d9de54  api 0.1.1 @ 14f37060eb1f


In [9]:
results = check_run(run_id, persist=True)

clean = sum(1 for _, v in results if v.outcome == "none")
print(f"{clean}/{len(results)} executions with no structural finding")

# Sorted so anything with a finding is at the top: a clean run is a long list
# of rows nobody needs to read, and the exceptions are the whole point.
tables.structural(results).sort_values(["findings", "kind"], ascending=[False, True])

28/29 executions with no structural finding


,,,kind,outcome,findings,detail
case,rep,attempt,,,,
venue_lookup_seafood_lisbon__dir4_not_a_meal,1,1,DIR-4,detected,1,the reply names no 'bar' place; it named ['res...
factual_qa_alfama_history__dir1_the_earthquake,1,1,DIR-1,none,0,
factual_qa_lisbon_november_weather__dir1_july_instead,1,1,DIR-1,none,0,
scheduling_move_tram__dir1_put_the_monastery_on,1,1,DIR-1,none,0,
venue_lookup_seafood_lisbon__dir1_and_a_hotel,1,1,DIR-1,none,0,
venue_lookup_sintra_lunch__dir1_coffee_not_lunch,1,1,DIR-1,none,0,
scheduling_belem_morning__dir2_on_the_9th,1,1,DIR-2,none,0,
scheduling_move_tram__dir2_sunday_afternoon,1,1,DIR-2,none,0,
venue_lookup_sintra_lunch__dir2_lisbon_day,1,1,DIR-2,none,0,


## Tool coverage for this run

Observed against declared. `all errored` is the column that
found the weather defect: a tool can be declared, called, and broken,
while every case stays green.

In [10]:
tables.tool_coverage(run_id)

,cases requiring,cases permitting,calls,errors,all errored,parameters exercised
tool,,,,,,
get_climate_normals,1,6,1,0,False,"include_marine, lat, lng"
get_region_climate,0,6,6,0,False,place
get_weather,2,13,3,1,False,"end_date, lat, lng, start_date"
move_slot,2,2,2,0,False,"day_date, end_time, slot_id, start_time"
propose_plan,3,3,3,0,False,"days, description, title"
read_wikipedia,3,3,6,0,False,"query, section"
schedule_block,1,1,1,0,False,"block_id, day_date, end_time, start_time"
search_places,10,10,23,0,False,"lat, lng, query"
unschedule_block,1,1,1,0,False,slot_id


## What the metric found

Two kinds of finding came out of this, and they are not the same kind of
evidence. The first is about the system under test and is what the pipeline
exists to produce; the second is about the instrument, and belongs in the
implementation log rather than in any claim about the agent.

**Defects in the agent and its tools** - the pipeline doing its job:

| found | how it surfaced | consequence |
|---|---|---|
| `get_weather` had **never once succeeded** in 118 recorded executions | coverage: declared, called, and every call errored | D56 - the prompt sent it at trip dates unconditionally, seven weeks past a sixteen-day horizon |
| The agent was **never told the current date** | a case that *required* `get_weather`'s success path, which nothing had before | D57 - "the next few days" had only the trip's dates to resolve against; it also accepted a date the member invented |
| The `get_region_climate` and `get_weather` instructions **contradicted** each other | the same case, answered with September averages for a trip three days away | D57 - the climate rule now defers inside the forecast horizon |
| An out-of-horizon call returned a bare `400`, and the agent **retried it** | tool error text in the trace | D56 - the tool now names the alternative, because its output is the agent's only view of it |
| Six of twelve tools were **never called** by any run | coverage, first pass | D50, D51 - extra seeds, each the only route to a tool |
| `get_region_climate` is required by **no case at all** | D66's disjunction fix | open - three weather cases would pass with it broken, by falling back to `get_climate_normals` |
| An empty reply conflated **three** different failures | the failure report's evidence columns | D59 - our deadline, a busy provider and a genuine agent failure now separate into 504, 503 and 502 |
| The capacity retry could **consume the whole turn** | latency split by `retry_count` | D60 - a 25s ceiling, so a turn gives up while it can still say so |

Measured effect of the weather fixes on the two cases that provoked them:
1/10 to 10/10 and 3/10 to 9/10, with `get_weather` making one call each and
**zero** errors - the first error-free calls of that tool in the record.

**Defects in the harness** - the instrument maturing, not the agent failing.
The reset was destroying the record it was meant to preserve (D38, L08); the
checker served one attempt's verdict to another (D65); coverage over-counted
disjunctive requirements (D66); scoring counted a superseded attempt (D59).
These are not evidence that the agent was broken and are not reported as such.
They are the reason §3.7 says an instrument of this kind needs its own tests.

## Did the perturbations actually perturb?

A directional relation claims the answer changes. The check above says whether
each execution matched its own constraints; this says whether the follow-up
differed from its seed at all — which is the part a passing verdict can hide.

A follow-up whose venue set is identical to its seed's has not been perturbed,
whatever its checks say.

In [11]:
# `overlap` is the column to read, and it means opposite things by kind:
# for a DIRECTIONAL relation a high overlap is a warning (the request changed
# and the answer did not), for an INVARIANCE relation it is the expected
# result. `suspect` flags only the first of those.
tables.perturbation(results)

,relation,kind,places,seed places,shared,overlap,tools,suspect
follow-up,,,,,,,,
advisory_packing_list__inv1_paraphrase,INV-1,invariance,0,0,0,NaN,same,False
advisory_packing_list__inv3_irrelevant_detail,INV-3,invariance,0,0,0,NaN,differ,False
advisory_sintra_full_or_half_day__inv1_paraphrase,INV-1,invariance,0,0,0,NaN,same,False
advisory_sintra_full_or_half_day__inv4_german,INV-4,invariance,0,0,0,NaN,same,False
factual_qa_alfama_history__dir1_the_earthquake,DIR-1,directional,0,0,0,NaN,same,False
factual_qa_alfama_history__inv1_paraphrase,INV-1,invariance,0,0,0,NaN,same,False
factual_qa_lisbon_november_weather__dir1_july_instead,DIR-1,directional,0,0,0,NaN,same,False
factual_qa_lisbon_november_weather__inv1_paraphrase,INV-1,invariance,0,0,0,NaN,same,False
scheduling_belem_morning__dir2_on_the_9th,DIR-2,directional,5,2,2,0.40,same,False


## Review

The cases above are a first set, run end to end. What to look at:

1. **Is the perturbed request what a member would actually type?** The wording
   is the whole input; a stilted one tests a sentence nobody writes.
2. **Is the claimed delta the right claim?** A directional case that asserts
   too much manufactures false regressions (D30); one that asserts too little
   passes whatever the agent does.
3. **The light cases** - factual QA's paraphrases and advisory's - assert
   nothing structurally. Worth deciding whether they earn their place or
   whether the budget is better spent elsewhere.
4. **`INV-2` (typos)** is expected to be the weakest test in the suite, and it
   is deliberately the only one of its relation (D61). It sits on the venue
   lookup because that is the closest the suite gets to a typo reaching a
   tool rather than only the model. The first run **falsified that mechanism**:
   the agent authors the `search_places` query and normalised every typo before
   the tool saw anything, so the follow-up searched a string byte-identical to
   the seed's and the case scored 10/10. The relation is weak, as predicted,
   but for a more specific reason than predicted - query rewriting, not
   tolerant matching downstream. Recording the expectation in advance is what
   made that distinguishable, and it narrows where to look if a typo case ever
   does fail.

Edits go in `cases/derivations.py` and this notebook is re-run; the YAML is
regenerated, never hand-edited.